# Validação espacial dos cenários LR

In [ ]:
import os
import sys

import pandas as pd

from glass.pys.oss import fprop
from glass.wt import obj_to_tbl

sys.path.append("/code/scripts")

from validation import pseudo_roc_blocks

In [ ]:
# ALTERAR APENAS ESTAS DUAS VARIÁVEIS

scenario = "C5"
area = "extremadura"


configs = {
    "C1": {
        "areas": ["centro"],
        "source": "icnf",
        "period": "1995-2024"
    },
    "C3": {
        "areas": ["centro"],
        "source": "icnf",
        "period": "2008-2024"
    },
    "C5": {
        "areas": ["centro", "extremadura"],
        "source": "effis",
        "period": "2008-2024"
    }
}


if scenario not in configs:
    raise ValueError(
        f"Cenário LR inválido: {scenario}"
    )


cfg = configs[scenario]

if area not in cfg["areas"]:
    raise ValueError(
        f"O cenário {scenario} não está disponível para {area}"
    )


source = cfg["source"]
ref_period = cfg["period"]
ref_file = ref_period.replace("-", "_")


base = f"/code/data/processed/{area}"
results = f"/code/data/results/{area}/{scenario}"
final_dir = f"{results}/final"
validation_dir = f"{results}/validation"

os.makedirs(
    validation_dir,
    exist_ok=True
)


# Referência retrospetiva.
refs = {
    f"success_rate_{ref_period}": (
        f"{base}/area_ardida/{source}/raster_binary/"
        f"rst_ba_{ref_file}_bin.tif"
    )
}


# C3 e C5 são avaliados com ambas as fontes de 2025
# para permitir a comparação do efeito da fonte.
if area == "centro" and scenario in ["C3", "C5"]:
    refs.update({
        "prediction_rate_2025_icnf": (
            f"{base}/area_ardida/icnf/raster_binary/"
            "rst_ba_2025_bin.tif"
        ),
        "prediction_rate_2025_effis": (
            f"{base}/area_ardida/effis/raster_binary/"
            "rst_ba_2025_bin.tif"
        )
    })

else:
    refs[f"prediction_rate_2025_{source}"] = (
        f"{base}/area_ardida/{source}/raster_binary/"
        "rst_ba_2025_bin.tif"
    )


products = {
    "dem": f"{final_dir}/lri_dem.tif",
    "slope": f"{final_dir}/lri_slope.tif",
    "lulc": f"{final_dir}/lri_lulc.tif",
    "susc": f"{final_dir}/res_lri.tif",
    "prob": f"{final_dir}/wprobability.tif",
    "peri": f"{final_dir}/res_perigosity.tif"
}


positive_value = 1

output_table = os.path.join(
    validation_dir,
    "lr_validation.xlsx"
)


print("Área:", area)
print("Cenário:", scenario)
print("Fonte de treino:", source)
print("Período:", ref_period)
print("Referências:", refs)
print("Produtos:", products)

In [ ]:
rows = []
curves = {}


for product, raster in products.items():
    raster_name = fprop(
        raster,
        "fn"
    )

    for evaluation, reference in refs.items():
        curve, auc_value, _ = pseudo_roc_blocks(
            ref=reference,
            perigo_rst=raster,
            posval=positive_value,
            otbl=None,
            block_size=1024
        )

        curve_name = (
            f"{scenario}_{product}_{evaluation}"
        )

        curve_file = os.path.join(
            validation_dir,
            f"{curve_name}_curve.xlsx"
        )

        obj_to_tbl(
            curve,
            curve_file
        )

        curves[curve_name] = curve

        rows.append({
            "area": area,
            "scenario": scenario,
            "method": "LR",
            "training_source": source,
            "training_period": ref_period,
            "product": product,
            "raster": raster_name,
            "evaluation": evaluation,
            "reference": reference,
            "auc": auc_value,
            "curve_file": curve_file
        })

        print(
            scenario,
            product,
            evaluation,
            round(auc_value, 6)
        )


validation_results = pd.DataFrame(
    rows
)

obj_to_tbl(
    validation_results,
    output_table
)

validation_results

In [ ]:
curve_name = (
    f"{scenario}_peri_"
    f"prediction_rate_2025_{source}"
)

curves[curve_name].plot.scatter(
    x="tarearatio",
    y="tfireration"
)